In [14]:
from huggingface_hub import notebook_login
# This will create an interactive login widget in your notebook
notebook_login()

In [18]:
import json
import logging
import re
import unicodedata
from huggingface_hub import HfFileSystem
from datasets import load_dataset

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("verify_overlap")

# Initialize Hugging Face File System to read Afrivoice manifests
fs = HfFileSystem()
afrivoice_repo_id = "datasets/DigitalUmuganda/Afrivoice"
waxal_repo_id = "google/WaxalNLP"

# -------------------------------------------------------------
# 1. TEXT NORMALIZATION (Adapted from verify_external_data.py)
# -------------------------------------------------------------
def loose_key(s: str) -> str:
    """
    Aggressive match key: NFKC, lowercase, punctuation->space, whitespace collapsed.
    Diacritics are KEPT (dropping them would create false-positive matches).
    """
    _PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
    s = unicodedata.normalize("NFKC", str(s)).lower()
    return " ".join(_PUNCT.sub(" ", s).split())

def _text_only(ds):
    """
    Disable Audio-column decoding on a streaming dataset so iterating reads only the 
    transcripts. This prevents `datasets` from eagerly decoding audio.
    """
    import datasets
    for col, feat in dict(getattr(ds, "features", None) or {}).items():
        if isinstance(feat, datasets.Audio):
            ds = ds.cast_column(col, datasets.Audio(decode=False))
    return ds

# -------------------------------------------------------------
# 2. LOAD ACTUAL WAXAL DATA FROM HUGGING FACE
# -------------------------------------------------------------
def build_waxal_refs(waxal_config: str):
    """
    Streams the actual WAXAL dataset splits to build reference sets for overlap checking.
    """
    logger.info(f"Building WAXAL references for '{waxal_config}'...")
    refs = {"train": set(), "val": set(), "test": set()}
    
    # WAXAL standard splits
    splits_to_check = [("train", "train"), ("validation", "val"), ("test", "test")]
    
    for hf_split, ref_key in splits_to_check:
        try:
            logger.info(f"Streaming WAXAL split: {hf_split}...")
            # Load dataset in streaming mode so we only pull text
            stream = load_dataset(waxal_repo_id, name=waxal_config, split=hf_split, streaming=True)
            stream = _text_only(stream)
            
            count = 0
            for ex in stream:
                text = ex.get("transcription", "")
                k = loose_key(text)
                if k:
                    refs[ref_key].add(k)
                    count += 1
            logger.info(f"Added {count} transcripts to {ref_key} references.")
        except Exception as e:
            logger.warning(f"Could not load {hf_split} split for {waxal_config}: {e}")
            
    return refs

# -------------------------------------------------------------
# 3. STREAM AFRIVOICE MANIFESTS 
# -------------------------------------------------------------
def stream_afrivoice_manifests(lang_folder: str):
    """
    Reads line-separated JSON records directly from the Afrivoice manifest files.
    """
    target_dir = f"{afrivoice_repo_id}/{lang_folder}"
    all_files = fs.ls(target_dir, detail=False)
    manifest_files = [f for f in all_files if "manifest_" in f and f.endswith(".json")]
    
    logger.info(f"Found {len(manifest_files)} manifest files for {lang_folder} in Afrivoice.")
    
    for manifest_path in sorted(manifest_files):
        with fs.open(manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                yield json.loads(line)

# -------------------------------------------------------------
# 4. OVERLAP CHECKER
# -------------------------------------------------------------
def check_overlap(afrivoice_lang: str, waxal_config: str):
    """
    Compares Afrivoice transcripts against the WAXAL dataset.
    """
    refs = build_waxal_refs(waxal_config)
    
    stats = {
        "total_scanned": 0,
        "test_breach_hits": 0,
        "train_hits": 0,
        "val_hits": 0,
        "breach_examples": []
    }
    
    logger.info(f"Scanning Afrivoice '{afrivoice_lang}' against WAXAL '{waxal_config}'...")
    stream = stream_afrivoice_manifests(afrivoice_lang)
    
    for record in stream:
        stats["total_scanned"] += 1
        text = record.get("transcription", "")
        k = loose_key(text)
        
        if not k:
            continue
            
        # WAXAL test transcripts -> ANY hit = breach material
        if k in refs["test"]:
            stats["test_breach_hits"] += 1
            if len(stats["breach_examples"]) < 5:
                stats["breach_examples"].append(text)
                
        # Train and Val checks
        if k in refs["train"]:
            stats["train_hits"] += 1
        if k in refs["val"]:
            stats["val_hits"] += 1
            
        if stats["total_scanned"] % 10000 == 0:
            logger.info(f"Scanned {stats['total_scanned']} records... Found {stats['test_breach_hits']} test breaches.")
            
    return stats

# -------------------------------------------------------------
# EXECUTION
# -------------------------------------------------------------
if __name__ == "__main__":
    # Check Lingala (WAXAL config: 'lin_asr')
    print("\n--- Checking Lingala ---")
    lingala_stats = check_overlap(afrivoice_lang="Lingala", waxal_config="lin_asr")
    print("\nLingala Results:\n", json.dumps(lingala_stats, indent=2))
    
    # Check Shona (WAXAL config: 'sna_asr')
    print("\n--- Checking Shona ---")
    shona_stats = check_overlap(afrivoice_lang="Shona", waxal_config="sna_asr")
    print("\nShona Results:\n", json.dumps(shona_stats, indent=2))

INFO:verify_overlap:Building WAXAL references for 'lin_asr'...
INFO:verify_overlap:Streaming WAXAL split: train...



--- Checking Lingala ---


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/WaxalNLP/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/google/WaxalNLP/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/WaxalNLP.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/google/WaxalNLP/google/WaxalNLP.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/WaxalNLP/revision/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.hugging

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/WaxalNLP/tree/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data%2FASR%2Flin?recursive=true&expand=false "HTTP/1.1 200 OK"


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

... (20 outputs left out)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/dataset_infos.json "HTTP/1.1 404 Not Found"


Resolving data files:   0%|          | 0/52 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/datasets/google/WaxalNLP/resolve/e0a62aaebc61bd5bb8cac17a08d1b42c65551dd2/data/ASR/sna/sna-test-00000.parquet "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/696e2528357a40707550b1c4/3ff1e9f29e802f54929f049ae0dcf7ed7180621aa9ed47f97d486adba102b725?Expires=1784454854&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY5NmUyNTI4MzU3YTQwNzA3NTUwYjFjNC8zZmYxZTlmMjllODAyZjU0OTI5ZjA0OWFlMGRjZjdlZDcxODA2MjFhYTllZDQ3Zjk3ZDQ4NmFkYmExMDJiNzI1KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDQ1NDg1NH19fV19&Signature=MEYCIQCiEb1ggM6TewSb1p-K53jNMvmTs5qz1k32zBmazCYA%7EwIhAJwj1XRLH1%7EiorA3Bxt3FFaKCGdKRCvjDPkC7iDMEz0%7E&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=69d3be4719133f17a1631d24&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27sna-test-00000.parquet%3B+filename%3D%22sna-test-00000.parquet%22%3B&X-Amz-Algorithm


Shona Results:
 {
  "total_scanned": 97240,
  "test_breach_hits": 1678,
  "train_hits": 13338,
  "val_hits": 1614,
  "breach_examples": [
    "Turusi bhokisi ine ruvara rutema ine maturusi akasiyana siyana. Maturusi arikuwanikamo anosanganisira mapinjisi nezvipanera. Zvipanera zvinoshandiswa kusunga nekusunungura simbi. Zvipanera izvi zvine ruvara rwesirivheri, zvimwewo ndizvo zvine mubato ine ruvara rutsvuku neruvara rutema. Koitawo pinjisi ine ruvara rwesirivheri kumusoro kuzasi ine ruvara rutema neruvara rutsvuku.",
    "Maturusi anoshandiswa mubindu anoti hara,foshoro,piki nebadza akazendamiswa pagaba.",
    "Chikafu chakagadzikwa pamusoro petafura. Icho chinova icho begger chakagadzirwa zvakaisvonaka chaizvo. Chakaiswawo zvinhu zvakaita semadomasi nenyama pakati. Uye nechepadivi pane bhotoro rine zvinwiwa. ",
    "Shangu mbiri dzinoratidza kuti imwe ndeye kuruboshwe imwe ndeye yekurudyi. Shangu idzi dzine ruvara rwebhuru. Nechepamusoro padzo panofekerwa bhutsu tichikomberedza dzi

In [20]:
shona_stats

{'total_scanned': 97240,
 'test_breach_hits': 1678,
 'train_hits': 13338,
 'val_hits': 1614,
 'breach_examples': ['Turusi bhokisi ine ruvara rutema ine maturusi akasiyana siyana. Maturusi arikuwanikamo anosanganisira mapinjisi nezvipanera. Zvipanera zvinoshandiswa kusunga nekusunungura simbi. Zvipanera izvi zvine ruvara rwesirivheri, zvimwewo ndizvo zvine mubato ine ruvara rutsvuku neruvara rutema. Koitawo pinjisi ine ruvara rwesirivheri kumusoro kuzasi ine ruvara rutema neruvara rutsvuku.',
  'Maturusi anoshandiswa mubindu anoti hara,foshoro,piki nebadza akazendamiswa pagaba.',
  'Chikafu chakagadzikwa pamusoro petafura. Icho chinova icho begger chakagadzirwa zvakaisvonaka chaizvo. Chakaiswawo zvinhu zvakaita semadomasi nenyama pakati. Uye nechepadivi pane bhotoro rine zvinwiwa. ',
  'Shangu mbiri dzinoratidza kuti imwe ndeye kuruboshwe imwe ndeye yekurudyi. Shangu idzi dzine ruvara rwebhuru. Nechepamusoro padzo panofekerwa bhutsu tichikomberedza dzinoratidza kuti dzine ruvara ruchena